In [7]:
!pip install requests beautifulsoup4 lxml loguru langchain-groq langchain_nvidia_ai_endpoints -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.8/61.8 kB 2.5 MB/s eta 0:00:00


In [8]:
import os
SEC_API_KEY_OLD = os.environ.get("SEC_API_KEY", "f589b2bc120b3ddb2d2835e86c694d83070a3445a74ee2d795f64abc2fa1370e")
SEC_API_KEY = os.environ.get("SEC_API_KEY", "c67770ee973fe24ae572720db4a152071d0f9c8eb1c2713b0cdc225089ad4989")
SEC_API_KEY = os.environ.get("SEC_API_KEY", "0e88bf462569faad922fa778a460da4bbe6d3d156f02075b64191e4d136b0fd0")




import os
import re
import json
import time
import hashlib
import logging
from datetime import datetime, timedelta
from html.parser import HTMLParser
from typing import Optional
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
from tqdm import tqdm
from dotenv import load_dotenv

load_dotenv(override=True)

load_dotenv(override=True)

# ─── NVIDIA LLM ─────────────────────────────────────────────────────────────────
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from google.colab import userdata

# Get API key from Colab secrets
nvidia_api_key = userdata.get('NVIDIA_API_KEY')  # or 'NVAPI_KEY'

MODEL = "moonshotai/kimi-k2.6"  # or "openai/gpt-oss-120b", "meta/llama3-70b", etc.

def get_llm(temperature: float = 0.2) -> ChatNVIDIA:
    return ChatNVIDIA(
        model=MODEL,
        api_key=nvidia_api_key,
        temperature=temperature,
        top_p=1,
        max_completion_tokens=16384,
    )




In [9]:
"""
SEC Fine-Tuning Dataset Builder  v8  — 100% FREE, PRODUCTION READY
===================================================================
Zero paid APIs. Pure SEC EDGAR. BeautifulSoup + lxml.

ROOT CAUSES FIXED vs v7 (diagnosed from actual output):
────────────────────────────────────────────────────────
BUG 1 — Wrong document selected (most critical)
  v7 used primaryDocument from submissions API.
  Problem: MSFT/AAPL file via Donnelley/Workiva → primaryDocument is a
  ~5KB cover/TOC shell htm. The actual 800KB narrative (MD&A, Risk Factors)
  is a SEPARATE file in the index (e.g. msft-20260331.htm).
  FIX: scan full filing index JSON, pick the LARGEST .htm by content-length
  header before downloading. Fall back to downloading top-3 candidates.

BUG 2 — 8-K gets XBRL cover shell, not earnings press release
  v7: primaryDocument for AAPL/MSFT 8-K = XBRL cover shell with bond
  member names and CIK strings. Zero financial content.
  FIX: in 8-K index, look for documents with type="EX-99.1" — this is
  where the actual earnings press release lives. Fall back to largest htm.

BUG 3 — Section regex finds TOC entry, not narrative
  v7: Item 7 regex matches the table-of-contents line "Item 7. MD&A ... 45"
  (720 chars including page number). MIN_SECTION_CHARS=300 lets it through.
  Then next item found immediately → empty section.
  FIX: require extracted section to contain ≥2 numeric dollar/percent
  patterns (real narrative always has numbers). TOC entries don't.

BUG 4 — Groq TPD limit: 200k/day → 60 calls max
  v7: openai/gpt-oss-120b has 200k TPD. 3 companies = 63 calls = exact hit.
  100 companies = 2100 calls = 35 days to complete.
  FIX: switch to meta-llama/llama-4-maverick-17b-128e-instruct (500k TPD)
  AND reduce output length to ~600-800 tokens (structured JSON, not prose).
  Result: 500k / 700 tokens avg = 714 calls/day → 100 companies in 3 days.

BUG 5 — Groq 429 not retried with sleep
  v7: on 429, generate_analysis() logs warning and returns "". Records saved
  without LLM output. No retry. Resume skips the ticker so they're lost.
  FIX: in generate_analysis(), detect 429 with sleep+retry (parse wait time
  from error message). Records without LLM output are written to a separate
  .pending file and retried at end of run.

BUG 6 — XMLParsedAsHTMLWarning flood (cosmetic but noisy)
  FIX: suppress once at module load with warnings.filterwarnings.

INSTALL (once per Colab session):
  pip install requests beautifulsoup4 lxml loguru langchain-groq

USAGE:
  Set GROQ_API_KEY in Colab secrets, then:
    tickers = get_nasdaq100_tickers()
    tickers = tickers[:5]   # test first
    build_dataset(tickers, OUTPUT_FILE, resume=True)
"""

import os
import re
import json
import sys
import time
import hashlib
import warnings
from datetime import datetime, timedelta
from typing import Optional

import requests
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning
from loguru import logger
from tqdm import tqdm

# Suppress BeautifulSoup XML-parsed-as-HTML warning (cosmetic)
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

# ─── LOGURU ───────────────────────────────────────────────────────────────────
logger.remove()
logger.add(
    sys.stderr,
    format="<green>{time:HH:mm:ss}</green> | <level>{level: <7}</level> | {message}",
    level="INFO",
    colorize=True,
)
logger.add(
    "sec_builder.log",
    format="{time:YYYY-MM-DD HH:mm:ss} | {level: <7} | {function}:{line} | {message}",
    level="DEBUG",
    rotation="50 MB",
    retention="7 days",
)


# ─── CONFIG ───────────────────────────────────────────────────────────────────
OUTPUT_FILE       = "sec_finetune_dataset.jsonl"
PENDING_FILE      = "sec_finetune_pending.jsonl"   # records needing LLM retry
YEARS_BACK        = 5
EDGAR_SLEEP       = 0.15     # SEC rate limit: ≤10 req/s
MAX_RETRIES       = 4
BACKOFF_BASE      = 2.0
PERIOD_FUZZ_DAYS  = 5        # ±days for XBRL period matching
MAX_TEXT_CHARS    = 5000     # per section
MIN_SECTION_CHARS = 400      # raised from 300 — filters out TOC entries better
# BUG 3 FIX: require section to contain real financial narrative
MIN_SECTION_NUMBERS = 2      # section must contain ≥N dollar/percent patterns
GROQ_429_MAX_WAIT   = 360    # max seconds to wait on 429 before giving up

EDGAR_HEADERS = {
    "User-Agent": "SECDatasetBuilder/1.0 (academic ML research; research@university.edu)",
    "Accept-Encoding": "gzip, deflate, br",
    "Accept": "application/json, text/html, */*",
}

# ─── SECTION PATTERNS ────────────────────────────────────────────────────────

SECTIONS_10K = {
    "business":             [r"item\s+1[\.\s]*[-–—]?\s*business\b",
                             r"(?<!\w)item\s+1(?!\s*[aAbB])\b"],
    "risk_factors":         [r"item\s+1a[\.\s]*[-–—]?\s*risk\s+factor",
                             r"item\s+1a\b"],
    "mda":                  [r"item\s+7[\.\s]*[-–—]?\s*management.{0,100}discussion",
                             r"(?<!\w)item\s+7(?!\s*a)\b"],
    "market_risk":          [r"item\s+7a[\.\s]*[-–—]?\s*quantitative",
                             r"item\s+7a\b"],
    "financial_statements": [r"item\s+8[\.\s]*[-–—]?\s*financial\s+statement",
                             r"(?<!\w)item\s+8\b"],
    "legal_proceedings":    [r"item\s+3[\.\s]*[-–—]?\s*legal\s+proceed",
                             r"(?<!\w)item\s+3\b"],
}

SECTIONS_10Q = {
    "mda":                  [r"item\s+2[\.\s]*[-–—]?\s*management.{0,100}discussion",
                             r"item\s+2\b"],
    "market_risk":          [r"item\s+3[\.\s]*[-–—]?\s*quantitative",
                             r"item\s+3\b"],
    "risk_factors":         [r"item\s+1a[\.\s]*[-–—]?\s*risk\s+factor",
                             r"item\s+1a\b"],
    "financial_statements": [r"item\s+1[\.\s]*[-–—]?\s*financial\s+statement",
                             r"(?<!\w)item\s+1(?!\s*a)\b"],
    "legal_proceedings":    [r"item\s+1[\.\s]*[-–—]?\s*legal\s+proceed"],
}

NEXT_ITEM_RE = re.compile(r"\bitem\s+\d{1,2}[ab]?\s*[\.\s]", re.IGNORECASE)

# BUG 3 FIX: pattern to detect real financial content vs TOC entries
_FINANCIAL_NUMBER_RE = re.compile(
    r"(\$[\d,.]+|\d+[\.,]\d+\s*%|\b\d{1,3}(?:,\d{3})+\b|\b\d+\.\d+\s*billion|\b\d+\.\d+\s*million)",
    re.IGNORECASE
)

# ─── iXBRL PREPROCESSING ────────────────────────────────────────────────────

_RE_IX_HIDDEN = re.compile(r"<ix:hidden[^>]*>.*?</ix:hidden\s*>", re.DOTALL | re.IGNORECASE)
_RE_IX_TAGS   = re.compile(r"</?ix:[A-Za-z][A-Za-z0-9]*(?:\s[^>]*)?>", re.IGNORECASE)
_RE_XBRL_TAGS = re.compile(r"</?xbrli?:[A-Za-z][A-Za-z0-9]*(?:\s[^>]*)?>", re.IGNORECASE)
_RE_LINK_TAGS = re.compile(r"</?link:[A-Za-z][A-Za-z0-9]*(?:\s[^>]*)?>", re.IGNORECASE)


def preprocess_ixbrl(html: str) -> str:
    """Strip XBRL namespace tags before HTML parsing. Keeps inner text content."""
    html = _RE_IX_HIDDEN.sub(" ", html)
    html = _RE_IX_TAGS.sub("",   html)
    html = _RE_XBRL_TAGS.sub("", html)
    html = _RE_LINK_TAGS.sub("", html)
    return html


def html_to_text(html: str) -> str:
    """iXBRL-aware HTML → plain text via BeautifulSoup+lxml."""
    try:
        html  = preprocess_ixbrl(html)
        soup  = BeautifulSoup(html, "lxml")
        for tag in soup(["script", "style", "head", "noscript", "meta", "link"]):
            tag.decompose()
        text = soup.get_text(separator="\n")
        text = re.sub(r"[ \t]{2,}", " ", text)
        text = re.sub(r"\n{3,}", "\n\n", text)
        return text.strip()
    except Exception as e:
        logger.warning(f"html_to_text fallback (regex): {e}")
        html = _RE_IX_HIDDEN.sub(" ", html)
        text = re.sub(r"<[^>]+>", " ", html)
        text = re.sub(r"[ \t]{2,}", " ", text)
        text = re.sub(r"\n{3,}", "\n\n", text)
        return text.strip()

# ─── TICKERS ──────────────────────────────────────────────────────────────────

def get_nasdaq100_tickers() -> list[dict]:
    return [
        {"ticker": "AAPL",  "name": "Apple Inc"},
        {"ticker": "MSFT",  "name": "Microsoft Corp"},
        {"ticker": "NVDA",  "name": "NVIDIA Corp"},
        {"ticker": "GOOGL", "name": "Alphabet Inc"},
        {"ticker": "AMZN",  "name": "Amazon.com Inc"},
        {"ticker": "META",  "name": "Meta Platforms Inc"},
        {"ticker": "TSLA",  "name": "Tesla Inc"},
        {"ticker": "AVGO",  "name": "Broadcom Inc"},
        {"ticker": "COST",  "name": "Costco Wholesale Corp"},
        {"ticker": "NFLX",  "name": "Netflix Inc"},
        {"ticker": "AMD",   "name": "Advanced Micro Devices"},
        {"ticker": "QCOM",  "name": "Qualcomm Inc"},
        {"ticker": "ADBE",  "name": "Adobe Inc"},
        {"ticker": "AMAT",  "name": "Applied Materials Inc"},
        {"ticker": "INTU",  "name": "Intuit Inc"},
        {"ticker": "MU",    "name": "Micron Technology Inc"},
        {"ticker": "LRCX",  "name": "Lam Research Corp"},
        {"ticker": "KLAC",  "name": "KLA Corp"},
        {"ticker": "PANW",  "name": "Palo Alto Networks"},
        {"ticker": "CRWD",  "name": "CrowdStrike Holdings"},
        {"ticker": "CSCO",  "name": "Cisco Systems Inc"},
        {"ticker": "PEP",   "name": "PepsiCo Inc"},
        {"ticker": "TMUS",  "name": "T-Mobile US Inc"},
        {"ticker": "ADP",   "name": "Automatic Data Processing"},
        {"ticker": "SBUX",  "name": "Starbucks Corp"},
        {"ticker": "GILD",  "name": "Gilead Sciences Inc"},
        {"ticker": "BKNG",  "name": "Booking Holdings Inc"},
        {"ticker": "ISRG",  "name": "Intuitive Surgical Inc"},
        {"ticker": "AMGN",  "name": "Amgen Inc"},
        {"ticker": "ADI",   "name": "Analog Devices Inc"},
        {"ticker": "TXN",   "name": "Texas Instruments Inc"},
        {"ticker": "MELI",  "name": "MercadoLibre Inc"},
        {"ticker": "PYPL",  "name": "PayPal Holdings Inc"},
        {"ticker": "ABNB",  "name": "Airbnb Inc"},
        {"ticker": "DASH",  "name": "DoorDash Inc"},
        {"ticker": "FTNT",  "name": "Fortinet Inc"},
        {"ticker": "NET",   "name": "Cloudflare Inc"},
        {"ticker": "MNST",  "name": "Monster Beverage Corp"},
        {"ticker": "ORLY",  "name": "O'Reilly Automotive Inc"},
        {"ticker": "PAYX",  "name": "Paychex Inc"},
        {"ticker": "PCAR",  "name": "PACCAR Inc"},
        {"ticker": "SPOT",  "name": "Spotify Technology SA"},
        {"ticker": "TSCO",  "name": "Tractor Supply Co"},
        {"ticker": "DLTR",  "name": "Dollar Tree Inc"},
        {"ticker": "EA",    "name": "Electronic Arts Inc"},
        {"ticker": "EBAY",  "name": "eBay Inc"},
        {"ticker": "FAST",  "name": "Fastenal Co"},
        {"ticker": "FISV",  "name": "Fiserv Inc"},
        {"ticker": "CDW",   "name": "CDW Corp"},
        {"ticker": "CEG",   "name": "Constellation Energy Corp"},
        {"ticker": "CDNS",  "name": "Cadence Design Systems"},
        {"ticker": "SNPS",  "name": "Synopsys Inc"},
        {"ticker": "MRVL",  "name": "Marvell Technology Inc"},
        {"ticker": "ROST",  "name": "Ross Stores Inc"},
        {"ticker": "ODFL",  "name": "Old Dominion Freight Line"},
        {"ticker": "CTAS",  "name": "Cintas Corp"},
        {"ticker": "VRSK",  "name": "Verisk Analytics Inc"},
        {"ticker": "CPRT",  "name": "Copart Inc"},
        {"ticker": "IDXX",  "name": "IDEXX Laboratories Inc"},
        {"ticker": "CTSH",  "name": "Cognizant Technology Solutions"},
        {"ticker": "DXCM",  "name": "DexCom Inc"},
        {"ticker": "BIIB",  "name": "Biogen Inc"},
        {"ticker": "ILMN",  "name": "Illumina Inc"},
        {"ticker": "REGN",  "name": "Regeneron Pharmaceuticals"},
        {"ticker": "VRTX",  "name": "Vertex Pharmaceuticals"},
        {"ticker": "MRNA",  "name": "Moderna Inc"},
        {"ticker": "ZS",    "name": "Zscaler Inc"},
        {"ticker": "OKTA",  "name": "Okta Inc"},
        {"ticker": "SNOW",  "name": "Snowflake Inc"},
        {"ticker": "DDOG",  "name": "Datadog Inc"},
        {"ticker": "MDB",   "name": "MongoDB Inc"},
        {"ticker": "TTD",   "name": "The Trade Desk Inc"},
        {"ticker": "TEAM",  "name": "Atlassian Corp"},
        {"ticker": "WDAY",  "name": "Workday Inc"},
        {"ticker": "VEEV",  "name": "Veeva Systems Inc"},
        {"ticker": "NOW",   "name": "ServiceNow Inc"},
        {"ticker": "CRM",   "name": "Salesforce Inc"},
        {"ticker": "ORCL",  "name": "Oracle Corp"},
        {"ticker": "IBM",   "name": "IBM Corp"},
        {"ticker": "HPQ",   "name": "HP Inc"},
        {"ticker": "DELL",  "name": "Dell Technologies Inc"},
        {"ticker": "STX",   "name": "Seagate Technology Holdings"},
        {"ticker": "WDC",   "name": "Western Digital Corp"},
        {"ticker": "NTAP",  "name": "NetApp Inc"},
        {"ticker": "KEYS",  "name": "Keysight Technologies Inc"},
        {"ticker": "NXPI",  "name": "NXP Semiconductors NV"},
        {"ticker": "MCHP",  "name": "Microchip Technology Inc"},
        {"ticker": "ON",    "name": "ON Semiconductor Corp"},
        {"ticker": "SWKS",  "name": "Skyworks Solutions Inc"},
        {"ticker": "MPWR",  "name": "Monolithic Power Systems"},
        {"ticker": "ENPH",  "name": "Enphase Energy Inc"},
        {"ticker": "ZM",    "name": "Zoom Video Communications"},
        {"ticker": "DOCU",  "name": "DocuSign Inc"},
        {"ticker": "ALGN",  "name": "Align Technology Inc"},
    ]

# ─── HTTP SESSION ─────────────────────────────────────────────────────────────

class EDGARSession:
    ARCHIVES = "https://www.sec.gov/Archives/edgar/data"
    DATA_BASE = "https://data.sec.gov"
    WWW_BASE  = "https://www.sec.gov"

    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update(EDGAR_HEADERS)

    def get(self, url: str, timeout: int = 30, is_html: bool = False,
            stream: bool = False) -> Optional[requests.Response]:
        time.sleep(EDGAR_SLEEP)
        hdrs = {**EDGAR_HEADERS,
                "Accept": "text/html,*/*;q=0.8"} if is_html else EDGAR_HEADERS
        for attempt in range(MAX_RETRIES):
            try:
                r = self.session.get(url, timeout=timeout, headers=hdrs, stream=stream)
                if r.status_code == 429:
                    wait = BACKOFF_BASE ** (attempt + 2)
                    logger.warning(f"429 → sleep {wait:.0f}s | {url[-70:]}")
                    time.sleep(wait)
                    continue
                if r.status_code in (403, 404):
                    logger.debug(f"{r.status_code} | {url[-70:]}")
                    return None
                r.raise_for_status()
                logger.debug(f"GET {r.status_code} {len(r.content):,}B | {url[-70:]}")
                return r
            except requests.exceptions.Timeout:
                time.sleep(BACKOFF_BASE ** (attempt + 1))
            except Exception as e:
                logger.error(f"GET error: {e} | {url[-70:]}")
                return None
        return None

    def head(self, url: str) -> int:
        """Returns Content-Length in bytes, or 0 on failure. Used for size-ranking."""
        time.sleep(EDGAR_SLEEP * 0.5)  # HEAD is cheap
        try:
            r = self.session.head(url, timeout=8, headers=EDGAR_HEADERS, allow_redirects=True)
            cl = r.headers.get("Content-Length", "0")
            return int(cl) if cl.isdigit() else 0
        except Exception:
            return 0

# ─── CIK LOOKUP ───────────────────────────────────────────────────────────────

_cik_cache: dict[str, str] = {}
_tickers_data: Optional[dict] = None

def ticker_to_cik(ticker: str, sess: EDGARSession) -> Optional[str]:
    global _tickers_data
    if ticker in _cik_cache:
        return _cik_cache[ticker]
    if _tickers_data is None:
        r = sess.get(f"{sess.WWW_BASE}/files/company_tickers.json")
        _tickers_data = r.json() if r else {}
        logger.info(f"CIK table: {len(_tickers_data):,} entries loaded")
    for entry in _tickers_data.values():
        if entry.get("ticker", "").upper() == ticker.upper():
            cik = str(entry["cik_str"]).zfill(10)
            _cik_cache[ticker] = cik
            return cik
    logger.warning(f"No CIK for {ticker}")
    return None

# ─── FILING DISCOVERY ────────────────────────────────────────────────────────

def get_filings(cik: str, form_type: str, sess: EDGARSession,
                max_results: int = 10) -> list[dict]:
    url = f"{sess.DATA_BASE}/submissions/CIK{cik}.json"
    r   = sess.get(url)
    if not r:
        return []
    data   = r.json()
    recent = data.get("filings", {}).get("recent", {})
    cutoff = (datetime.now() - timedelta(days=YEARS_BACK * 365)).strftime("%Y-%m-%d")

    all_rows = list(zip(
        recent.get("form", []), recent.get("filingDate", []),
        recent.get("reportDate", []), recent.get("accessionNumber", []),
        recent.get("primaryDocument", []),
    ))
    for ef in data.get("filings", {}).get("files", [])[:5]:
        fname = ef.get("name", "")
        if not fname: continue
        r2 = sess.get(f"{sess.DATA_BASE}/submissions/{fname}")
        if not r2: continue
        try:
            ed = r2.json()
            all_rows += list(zip(
                ed.get("form",[]), ed.get("filingDate",[]),
                ed.get("reportDate",[]), ed.get("accessionNumber",[]),
                ed.get("primaryDocument",[]),
            ))
        except Exception: pass

    base = form_type.split("/")[0]
    results = []
    for form, dt, period, accno, doc in all_rows:
        if base not in form: continue
        if dt < cutoff: continue
        results.append({"form_type": form, "filed_date": dt,
                        "period_of_report": period, "accession_no": accno,
                        "primary_document": doc})
        if len(results) >= max_results:
            break
    logger.info(f"  {form_type}: {len(results)} filings")
    return results

# ─── DOCUMENT URL RESOLUTION (BUG 1 + 2 FIX) ─────────────────────────────────

def _get_index_docs(cik: str, accession_no: str, sess: EDGARSession) -> list[dict]:
    """Fetches the filing index and returns list of document dicts."""
    clean = accession_no.replace("-", "")
    cik_i = int(cik)
    idx   = f"{sess.ARCHIVES}/{cik_i}/{clean}/{accession_no}-index.json"
    r     = sess.get(idx)
    if not r:
        return []
    try:
        return r.json().get("documents", [])
    except Exception:
        return []


def _is_xbrl_viewer_stub(filename: str) -> bool:
    """Returns True for XBRL viewer stubs (R1.htm, R2.htm, ...) which have no narrative."""
    return bool(re.match(r"^R\d+\.htm$", filename, re.IGNORECASE))


def get_narrative_doc(cik: str, accession_no: str, primary_doc: str,
                      sess: EDGARSession, form_type: str = "") -> tuple[Optional[str], str]:
    """
    BUG 1 FIX: Selects the LARGEST narrative .htm from the filing index.

    Strategy (in order):
    1. Fast path: if primaryDocument is large (>50KB), use it directly.
       This handles companies that self-file (NVDA, some others).
    2. Index scan: fetch filing index JSON, rank all .htm candidates by
       Content-Length header (HEAD requests), download the largest.
       The largest .htm is almost always the complete narrative document.
    3. Fallback: try primaryDocument anyway if nothing else worked.

    Returns (url, html_text) or (None, "").
    """
    clean = accession_no.replace("-", "")
    cik_i = int(cik)
    base  = f"{sess.ARCHIVES}/{cik_i}/{clean}"

    # ── Fast path: check if primaryDocument is already the full filing ────
    if primary_doc and primary_doc.lower().endswith(".htm"):
        url = f"{base}/{primary_doc}"
        r   = sess.get(url, timeout=60, is_html=True)
        if r and len(r.content) > 50_000:
            logger.debug(f"  Doc (fast path, {len(r.content)//1024}KB): {primary_doc}")
            return url, r.text

    # ── Index scan: rank candidates by size ───────────────────────────────
    docs = _get_index_docs(cik, accession_no, sess)
    if not docs:
        logger.warning(f"  No index docs for {accession_no}")
        return None, ""

    candidates = []
    for doc in docs:
        dfile = doc.get("document", "")
        dtype = doc.get("type",     "")
        if not dfile.lower().endswith(".htm"):
            continue
        if _is_xbrl_viewer_stub(dfile):
            continue
        # Skip known cover/shell patterns
        if any(pat in dfile.lower() for pat in ["cover", "signature"]):
            continue
        candidates.append({"file": dfile, "type": dtype})

    if not candidates:
        logger.warning(f"  No htm candidates in index for {accession_no}")
        return None, ""

    # Rank by Content-Length (HEAD requests — fast, no body download)
    sized = []
    for c in candidates:
        url = f"{base}/{c['file']}"
        sz  = sess.head(url)
        sized.append((sz, url, c))
        logger.debug(f"    HEAD {sz//1024}KB — {c['file']}")

    # Sort largest first
    sized.sort(key=lambda x: x[0], reverse=True)

    # Try top-3 by size
    for sz, url, c in sized[:3]:
        r = sess.get(url, timeout=90, is_html=True)
        if r and len(r.content) > 20_000:
            logger.info(f"  Doc ({sz//1024}KB, {c['type'] or 'unknown'}): {c['file']}")
            return url, r.text

    logger.warning(f"  All candidates failed for {accession_no}")
    return None, ""


def get_8k_exhibit(cik: str, accession_no: str, primary_doc: str,
                   sess: EDGARSession) -> tuple[Optional[str], str]:
    """
    BUG 2 FIX: For 8-K filings, look for EX-99.1 (earnings press release).

    The primaryDocument for most 8-K filings is an XBRL cover shell with
    bond names and CIK strings — zero financial content. The actual earnings
    press release is always in the EX-99.1 exhibit.

    Falls back to largest htm if no EX-99.1 found.
    """
    clean = accession_no.replace("-", "")
    cik_i = int(cik)
    base  = f"{sess.ARCHIVES}/{cik_i}/{clean}"

    docs = _get_index_docs(cik, accession_no, sess)

    # Priority 1: EX-99.1 (earnings press release)
    for doc in docs:
        dtype = doc.get("type", "")
        dfile = doc.get("document", "")
        if "EX-99" in dtype.upper() and dfile.lower().endswith(".htm"):
            url = f"{base}/{dfile}"
            r   = sess.get(url, timeout=60, is_html=True)
            if r and len(r.content) > 3_000:
                logger.info(f"  8-K EX-99.1 found: {dfile} ({len(r.content)//1024}KB)")
                return url, r.text

    # Priority 2: any EX-99 variant
    for doc in docs:
        dtype = doc.get("type", "")
        dfile = doc.get("document", "")
        if "EX-99" in dtype.upper() and (dfile.endswith(".htm") or dfile.endswith(".txt")):
            url = f"{base}/{dfile}"
            r   = sess.get(url, timeout=60, is_html=True)
            if r and len(r.content) > 3_000:
                logger.info(f"  8-K {dtype} found: {dfile}")
                return url, r.text

    # Fallback: largest htm
    candidates = [(doc.get("document",""), doc.get("type",""))
                  for doc in docs
                  if doc.get("document","").lower().endswith(".htm")
                  and not _is_xbrl_viewer_stub(doc.get("document",""))]

    sized = []
    for dfile, dtype in candidates:
        url = f"{base}/{dfile}"
        sz  = sess.head(url)
        sized.append((sz, url))

    sized.sort(reverse=True)
    for sz, url in sized[:2]:
        r = sess.get(url, timeout=60, is_html=True)
        if r and len(r.content) > 3_000:
            logger.info(f"  8-K fallback ({sz//1024}KB): {url.split('/')[-1]}")
            return url, r.text

    logger.warning(f"  No usable 8-K document for {accession_no}")
    return None, ""

# ─── SECTION EXTRACTION (BUG 3 FIX) ─────────────────────────────────────────

def _has_financial_numbers(text: str) -> bool:
    """
    BUG 3 FIX: Returns True if text contains real financial numbers.
    Filters out TOC entries that regex-match the item heading but have no content.
    """
    return len(_FINANCIAL_NUMBER_RE.findall(text)) >= MIN_SECTION_NUMBERS


def extract_section(text: str, patterns: list[str]) -> str:
    """
    Regex section extractor with quality gate.

    Two-pass strategy:
    1. Find first pattern match in the text.
    2. Scan forward for the next Item heading to bound the section.
    3. Quality gate: reject if too short OR contains no financial numbers.
    4. If rejected, try the next occurrence of the same pattern (handles
       TOC entries that appear before the real section).
    """
    lower = text.lower()

    for search_start in range(0, len(lower), 1):
        # Find pattern start
        start = -1
        for pat in patterns:
            m = re.search(pat, lower[search_start:], re.DOTALL)
            if m:
                start = search_start + m.start()
                break
        if start < 0:
            break

        # Find end boundary
        tail_start = start + 30
        end_m      = NEXT_ITEM_RE.search(lower[tail_start:])
        end        = (tail_start + end_m.start()) if end_m else min(start + MAX_TEXT_CHARS * 2, len(text))

        section = text[start:end].strip()

        # Quality gate
        if len(section) >= MIN_SECTION_CHARS and _has_financial_numbers(section):
            return section[:MAX_TEXT_CHARS]

        # This match was a TOC entry — advance past it and try again
        search_start = start + 1
        if search_start >= len(lower):
            break

    return ""


def scrape_text_sections(cik: str, accession_no: str, primary_doc: str,
                          form_type: str, sess: EDGARSession) -> dict[str, str]:
    """Full pipeline: fetch → iXBRL strip → BeautifulSoup → section regex."""
    url, html = get_narrative_doc(cik, accession_no, primary_doc, sess, form_type)
    if not html:
        logger.warning(f"  No HTML retrieved for {accession_no}")
        return {}

    plain = html_to_text(html)
    logger.debug(f"  Plain text: {len(plain):,} chars")

    section_map = SECTIONS_10K if "10-K" in form_type else SECTIONS_10Q
    result = {}
    for name, pats in section_map.items():
        text = extract_section(plain, pats)
        result[name] = text
        status = f"{len(text):,} chars ✓" if text else "not found ✗"
        logger.debug(f"    {name}: {status}")

    filled = sum(1 for v in result.values() if v)
    logger.info(f"    Text sections: {filled}/{len(section_map)} filled")
    return result


def scrape_8k_text(cik: str, accession_no: str, primary_doc: str,
                   sess: EDGARSession) -> str:
    """Fetch 8-K EX-99.1 press release text."""
    _, html = get_8k_exhibit(cik, accession_no, primary_doc, sess)
    if not html:
        return ""
    plain = html_to_text(html)
    return plain[:4000]

# ─── XBRL METRICS ────────────────────────────────────────────────────────────

def load_xbrl_facts(cik: str, sess: EDGARSession) -> dict:
    url = f"{sess.DATA_BASE}/api/xbrl/companyfacts/CIK{cik}.json"
    r   = sess.get(url, timeout=30)
    if not r:
        return {}
    facts = r.json().get("facts", {})
    gaap  = facts.get("us-gaap", {})
    dei   = facts.get("dei", {})
    logger.info(f"  XBRL: {len(gaap)} GAAP + {len(dei)} DEI")
    return {"us-gaap": gaap, "dei": dei}


def xbrl_val(ns: dict, concepts: list[str], period_end: str, base_form: str) -> Optional[float]:
    try:
        td     = datetime.strptime(period_end, "%Y-%m-%d")
        window = {(td + timedelta(days=d)).strftime("%Y-%m-%d")
                  for d in range(-PERIOD_FUZZ_DAYS, PERIOD_FUZZ_DAYS + 1)}
    except Exception:
        window = {period_end}
    for concept in concepts:
        for unit in ("USD", "USD/shares", "shares", "pure"):
            for entry in ns.get(concept, {}).get("units", {}).get(unit, []):
                if (entry.get("end") in window
                        and entry.get("form", "").startswith(base_form)
                        and entry.get("val") is not None):
                    try:
                        return float(entry["val"])
                    except (ValueError, TypeError):
                        pass
    return None


def extract_metrics(xbrl: dict, period_end: str, form_type: str) -> dict:
    gaap      = xbrl.get("us-gaap", {})
    dei       = xbrl.get("dei", {})
    base_form = form_type.split("/")[0]
    g  = lambda ns, *c: xbrl_val(ns, list(c), period_end, base_form)

    m = {
        "revenue":                   g(gaap, "RevenueFromContractWithCustomerExcludingAssessedTax",
                                             "RevenueFromContractWithCustomerIncludingAssessedTax",
                                             "Revenues", "SalesRevenueNet"),
        "cost_of_revenue":           g(gaap, "CostOfGoodsAndServicesSold","CostOfRevenue","CostOfGoodsSold"),
        "gross_profit":              g(gaap, "GrossProfit"),
        "operating_expenses":        g(gaap, "OperatingExpenses"),
        "research_development":      g(gaap, "ResearchAndDevelopmentExpense"),
        "selling_general_admin":     g(gaap, "SellingGeneralAndAdministrativeExpense"),
        "operating_income":          g(gaap, "OperatingIncomeLoss"),
        "interest_expense":          g(gaap, "InterestExpense","InterestAndDebtExpense"),
        "pretax_income":             g(gaap, "IncomeLossFromContinuingOperationsBeforeIncomeTaxesExtraordinaryItemsNoncontrollingInterest"),
        "income_tax":                g(gaap, "IncomeTaxExpenseBenefit"),
        "net_income":                g(gaap, "NetIncomeLoss"),
        "eps_basic":                 g(gaap, "EarningsPerShareBasic"),
        "eps_diluted":               g(gaap, "EarningsPerShareDiluted"),
        "shares_basic":              g(gaap, "WeightedAverageNumberOfSharesOutstandingBasic"),
        "shares_diluted":            g(gaap, "WeightedAverageNumberOfDilutedSharesOutstanding"),
        "total_assets":              g(gaap, "Assets"),
        "current_assets":            g(gaap, "AssetsCurrent"),
        "total_liabilities":         g(gaap, "Liabilities"),
        "current_liabilities":       g(gaap, "LiabilitiesCurrent"),
        "total_equity":              g(gaap, "StockholdersEquity",
                                           "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest"),
        "cash_and_equivalents":      g(gaap, "CashAndCashEquivalentsAtCarryingValue",
                                           "CashCashEquivalentsAndShortTermInvestments",
                                           "CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalents"),
        "short_term_investments":    g(gaap, "ShortTermInvestments","MarketableSecuritiesCurrent"),
        "accounts_receivable":       g(gaap, "AccountsReceivableNetCurrent"),
        "inventory":                 g(gaap, "InventoryNet"),
        "goodwill":                  g(gaap, "Goodwill"),
        "intangible_assets":         g(gaap, "FiniteLivedIntangibleAssetsNet",
                                           "IntangibleAssetsNetExcludingGoodwill"),
        "long_term_debt":            g(gaap, "LongTermDebt","LongTermDebtNoncurrent"),
        "short_term_debt":           g(gaap, "ShortTermBorrowings","LongTermDebtCurrent"),
        "retained_earnings":         g(gaap, "RetainedEarningsAccumulatedDeficit"),
        "operating_cash_flow":       g(gaap, "NetCashProvidedByUsedInOperatingActivities"),
        "investing_cash_flow":       g(gaap, "NetCashProvidedByUsedInInvestingActivities"),
        "financing_cash_flow":       g(gaap, "NetCashProvidedByUsedInFinancingActivities"),
        "capex":                     g(gaap, "PaymentsToAcquirePropertyPlantAndEquipment"),
        "depreciation_amortization": g(gaap, "DepreciationDepletionAndAmortization",
                                           "DepreciationAndAmortization"),
        "stock_based_compensation":  g(gaap, "ShareBasedCompensation",
                                           "AllocatedShareBasedCompensationExpense"),
        "dividends_paid":            g(gaap, "PaymentsOfDividends","PaymentsOfDividendsCommonStock"),
        "share_repurchases":         g(gaap, "PaymentsForRepurchaseOfCommonStock"),
        "employees":                 g(dei,  "EntityNumberOfEmployees"),
        "entity_public_float":       g(dei,  "EntityPublicFloat"),
    }

    # Derived
    ocf, capex = m.get("operating_cash_flow"), m.get("capex")
    if ocf is not None and capex is not None:
        m["free_cash_flow"] = ocf - abs(capex)
    oi, da = m.get("operating_income"), m.get("depreciation_amortization")
    if oi is not None and da is not None:
        m["ebitda"] = oi + abs(da)

    rev, gp  = m.get("revenue"), m.get("gross_profit")
    ni,  ta  = m.get("net_income"), m.get("total_assets")
    eq,  dbt = m.get("total_equity"), m.get("long_term_debt")
    csh, oi2 = m.get("cash_and_equivalents"), m.get("operating_income")

    if rev and rev > 0:
        if gp  is not None: m["gross_margin_pct"]     = round(gp  / rev * 100, 2)
        if ni  is not None: m["net_margin_pct"]       = round(ni  / rev * 100, 2)
        if oi2 is not None: m["operating_margin_pct"] = round(oi2 / rev * 100, 2)
    if ta and ta > 0 and ni is not None: m["return_on_assets_pct"] = round(ni / ta * 100, 2)
    if eq and eq > 0 and ni is not None: m["return_on_equity_pct"] = round(ni / eq * 100, 2)
    if dbt is not None and csh is not None: m["net_debt"] = dbt - csh
    if eq and eq > 0 and dbt is not None:   m["debt_to_equity"]    = round(dbt / eq, 3)
    ca, cl = m.get("current_assets"), m.get("current_liabilities")
    if ca and cl and cl > 0: m["current_ratio"] = round(ca / cl, 3)

    clean = {k: (round(v, 4) if isinstance(v, float) and abs(v) > 0.01 else v)
             for k, v in m.items() if v is not None}
    logger.debug(f"    Metrics: {len(clean)} fields")
    return clean

# ─── YoY CHANGES & SIGNALS ───────────────────────────────────────────────────

def compute_changes(current: dict, previous: dict) -> dict:
    PAIRS = [
        ("revenue","revenue_yoy_pct"), ("net_income","net_income_yoy_pct"),
        ("gross_profit","gross_profit_yoy_pct"), ("operating_income","operating_income_yoy_pct"),
        ("ebitda","ebitda_yoy_pct"), ("free_cash_flow","fcf_yoy_pct"),
        ("operating_cash_flow","ocf_yoy_pct"), ("long_term_debt","debt_yoy_pct"),
        ("total_assets","assets_yoy_pct"), ("total_equity","equity_yoy_pct"),
        ("research_development","rd_yoy_pct"), ("eps_diluted","eps_yoy_pct"),
    ]
    c = {}
    for metric, label in PAIRS:
        cur, prev = current.get(metric), previous.get(metric)
        if cur is not None and prev and prev != 0:
            c[label] = round(((cur - prev) / abs(prev)) * 100, 2)
    for margin in ("gross_margin_pct","net_margin_pct","operating_margin_pct"):
        cur, prev = current.get(margin), previous.get(margin)
        if cur is not None and prev is not None:
            c[f"{margin}_delta"] = round(cur - prev, 2)

    rev = c.get("revenue_yoy_pct")
    if rev is not None:
        c["trend_revenue"] = (
            "hypergrowth" if rev > 50 else "strong_growth" if rev > 20 else
            "moderate_growth" if rev > 5 else "slowing_growth" if rev > 0 else
            "mild_decline" if rev > -10 else "significant_decline")
    gm = c.get("gross_margin_pct_delta")
    if gm is not None:
        c["trend_margin"] = "expanding" if gm > 1 else "compressing" if gm < -1 else "stable"
    fcf = c.get("fcf_yoy_pct")
    if fcf is not None:
        c["trend_fcf"] = "strong_generation" if fcf > 20 else "improving" if fcf > 0 else "deteriorating"
    dbt = c.get("debt_yoy_pct")
    if dbt is not None:
        c["trend_leverage"] = (
            "rapidly_increasing" if dbt > 30 else "moderately_increasing" if dbt > 10 else
            "stable" if -10 <= dbt <= 10 else "decreasing")
    return c


def compute_signals(m: dict, c: dict) -> dict:
    s = {}
    nm = m.get("net_margin_pct", 0)
    s["profitability_tier"] = (
        "exceptional" if nm > 30 else "strong" if nm > 15 else
        "average" if nm > 5 else "weak" if nm > 0 else "loss_making")
    ni  = m.get("net_income",    0) or 0
    fcf = m.get("free_cash_flow",0) or 0
    if ni > 0:
        r = fcf / ni
        s["cash_conversion"] = "excellent" if r > 1.1 else "good" if r > 0.8 else "moderate" if r > 0.5 else "poor"
    dr, cr = m.get("debt_to_equity"), m.get("current_ratio")
    if dr is not None:
        s["leverage_health"] = (
            "minimal_debt" if dr < 0.3 else "conservative" if dr < 1.0 else
            "moderate_leverage" if dr < 2.0 else "high_leverage" if dr < 4.0 else "overleveraged")
    if cr is not None:
        s["liquidity_health"] = (
            "very_strong" if cr > 3.0 else "strong" if cr > 2.0 else "adequate" if cr > 1.0 else "tight")
    rev_g = c.get("revenue_yoy_pct", 0) or 0
    op_m  = m.get("operating_margin_pct", 0) or 0
    r40   = rev_g + op_m
    s["rule_of_40_score"] = round(r40, 1)
    s["rule_of_40_pass"]  = r40 >= 40
    rev = m.get("revenue", 0) or 0
    rd  = m.get("research_development", 0) or 0
    if rev > 0 and rd > 0:
        rdi = rd / rev * 100
        s["rd_intensity_pct"]     = round(rdi, 2)
        s["innovation_investment"] = (
            "heavy" if rdi > 20 else "moderate" if rdi > 10 else "light" if rdi > 3 else "minimal")
    return s

# ─── LLM GENERATION (BUG 4 + 5 FIX) ─────────────────────────────────────────

RISK_KW = [
    "competition","regulation","supply chain","cybersecurity","inflation",
    "interest rate","liquidity","litigation","currency risk","macroeconomic",
    "recession","geopolitical","tariff","sanctions","climate","data privacy",
    "customer concentration","patent","labor shortage","ai disruption",
]

def _usd(v) -> str:
    if v is None: return "N/A"
    if abs(v) >= 1e12: return f"${v/1e12:.2f}T"
    if abs(v) >= 1e9:  return f"${v/1e9:.1f}B"
    if abs(v) >= 1e6:  return f"${v/1e6:.0f}M"
    return f"${v:,.0f}"

def _pct(v, sfx="%") -> str:
    return f"{v:+.1f}{sfx}" if v is not None else "N/A"


def build_prompt(record: dict) -> str:
    """
    BUG 4 FIX: Structured JSON output instead of 5-section prose.
    Reduces output from ~1600 tokens to ~600-700 tokens.
    This triples the number of records processable per day within the TPD limit.
    """
    m, c, s   = record.get("metrics",{}), record.get("changes",{}), record.get("signals",{})
    txt, meta = record.get("text_sections",{}), record.get("meta",{})
    has_text  = any(v for v in txt.values() if v and len(v) > 100)
    note      = "" if has_text else "\nNOTE: Text sections unavailable. Use quantitative data only. Do NOT fabricate.\n"

    return f"""You are a senior equity research analyst. Analyze this {meta.get('form_type')} filing.
Company: {meta.get('company_name')} ({meta.get('ticker')}) | Period: {meta.get('period_of_report')} | Filed: {meta.get('filed_date')}{note}

FINANCIALS
Revenue: {_usd(m.get('revenue'))} (YoY: {_pct(c.get('revenue_yoy_pct'))}) | Gross Margin: {_pct(m.get('gross_margin_pct'))} Δ{_pct(c.get('gross_margin_pct_delta'),'pp')} | Op Margin: {_pct(m.get('operating_margin_pct'))}
Net Income: {_usd(m.get('net_income'))} (YoY: {_pct(c.get('net_income_yoy_pct'))}) | EPS: {m.get('eps_diluted','N/A')} (YoY: {_pct(c.get('eps_yoy_pct'))})
FCF: {_usd(m.get('free_cash_flow'))} (YoY: {_pct(c.get('fcf_yoy_pct'))}) | OCF: {_usd(m.get('operating_cash_flow'))} | CapEx: {_usd(m.get('capex'))}
Cash: {_usd(m.get('cash_and_equivalents'))} | LT Debt: {_usd(m.get('long_term_debt'))} (YoY: {_pct(c.get('debt_yoy_pct'))}) | D/E: {m.get('debt_to_equity','N/A')} | CR: {m.get('current_ratio','N/A')}
R&D: {_usd(m.get('research_development'))} ({_pct(s.get('rd_intensity_pct'))} of rev) | SBC: {_usd(m.get('stock_based_compensation'))} | Buybacks: {_usd(m.get('share_repurchases'))}
EBITDA: {_usd(m.get('ebitda'))} | ROE: {_pct(m.get('return_on_equity_pct'))} | Employees: {m.get('employees','N/A')}
Signals: {s.get('profitability_tier','N/A')} profitability | {s.get('leverage_health','N/A')} leverage | Rule-of-40: {s.get('rule_of_40_score','N/A')} ({'PASS' if s.get('rule_of_40_pass') else 'FAIL'})

FILING TEXT
MD&A: {(txt.get('mda','') or '')[:900] or '[not available]'}
Risk Factors: {(txt.get('risk_factors','') or '')[:500] or '[not available]'}
Business: {(txt.get('business','') or '')[:300] or '[not available]'}

Respond with ONLY a JSON object (no markdown, no prose outside JSON):
{{
  "revenue_analysis": "1-2 sentences on revenue trend with exact numbers",
  "profitability": "1-2 sentences on margins and net income with exact numbers",
  "cash_flow": "1-2 sentences on FCF quality and capital allocation",
  "balance_sheet": "1 sentence on debt/liquidity position",
  "key_risks": ["risk1", "risk2", "risk3"],
  "investment_signal": "STRONG_BUY | BUY | HOLD | REDUCE | SELL",
  "signal_rationale": "1 sentence explaining the signal"
}}"""


def _parse_429_wait(error_msg: str) -> int:
    """Extracts wait time in seconds from Groq 429 error message."""
    # "Please try again in 4m29.136s"
    m = re.search(r"try again in (\d+)m([\d.]+)s", error_msg)
    if m:
        return int(m.group(1)) * 60 + int(float(m.group(2))) + 5
    m = re.search(r"try again in ([\d.]+)s", error_msg)
    if m:
        return int(float(m.group(1))) + 5
    return 60


def generate_analysis(record: dict, llm) -> str:
    """
    BUG 5 FIX: Retry on 429 with proper sleep duration parsed from error.
    Returns empty string only after exhausting retries.
    """
    for attempt in range(3):
        try:
            resp = llm.invoke(build_prompt(record))
            text = resp.content.strip()
            # Validate it's parseable JSON
            try:
                json.loads(text)
                return text
            except json.JSONDecodeError:
                # Extract JSON from markdown code block if LLM wrapped it
                m = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
                if m:
                    try:
                        json.loads(m.group(1))
                        return m.group(1)
                    except Exception:
                        pass
                # Return as-is — downstream handles non-JSON gracefully
                return text
        except Exception as e:
            err = str(e)
            if "429" in err or "rate_limit" in err.lower():
                wait = _parse_429_wait(err)
                if wait > GROQ_429_MAX_WAIT:
                    logger.warning(f"  429: wait {wait}s exceeds limit — skipping LLM")
                    return ""
                logger.warning(f"  429 → sleeping {wait}s (attempt {attempt+1}/3)")
                time.sleep(wait)
                continue
            logger.warning(f"  LLM error: {e}")
            return ""
    return ""


def build_instruction_pair(record: dict, analysis: str) -> dict:
    m, c, s   = record.get("metrics",{}), record.get("changes",{}), record.get("signals",{})
    txt, meta = record.get("text_sections",{}), record.get("meta",{})
    has_text  = any(v for v in txt.values() if v and len(v) > 100)
    risks     = [kw for kw in RISK_KW
                 if kw in ((txt.get("risk_factors","") or "") + (txt.get("mda","") or "")).lower()][:8]

    inp = (
        f"Provide a structured financial analysis for {meta.get('company_name')} "
        f"({meta.get('ticker')}) — {meta.get('form_type')} ending {meta.get('period_of_report')}.\n\n"
        f"Revenue: {_usd(m.get('revenue'))} | NI: {_usd(m.get('net_income'))} | FCF: {_usd(m.get('free_cash_flow'))}\n"
        f"Gross Margin: {m.get('gross_margin_pct','N/A')}% | Net Margin: {m.get('net_margin_pct','N/A')}% | ROE: {m.get('return_on_equity_pct','N/A')}%\n"
        f"Revenue YoY: {c.get('revenue_yoy_pct','N/A')}% | EPS: {m.get('eps_diluted','N/A')} (YoY: {c.get('eps_yoy_pct','N/A')}%)\n"
        f"D/E: {m.get('debt_to_equity','N/A')} | CR: {m.get('current_ratio','N/A')} | Rule-of-40: {s.get('rule_of_40_score','N/A')}\n"
    )
    if has_text:
        inp += f"\nMD&A excerpt: {(txt.get('mda','') or '')[:400]}\n"
        inp += f"Risk excerpt: {(txt.get('risk_factors','') or '')[:200]}"
    else:
        inp += "\n[Quantitative analysis only — text sections not available]"

    return {
        "instruction": (
            "You are a senior financial analyst. Analyze the SEC filing data and "
            "return a structured JSON financial assessment."
        ),
        "input":  inp,
        "output": analysis,
        "metadata": {
            "ticker":               meta.get("ticker"),
            "company":              meta.get("company_name"),
            "period":               meta.get("period_of_report"),
            "form_type":            meta.get("form_type"),
            "analysis_mode":        "full" if has_text else "quantitative_only",
            "profitability":        s.get("profitability_tier"),
            "trend_revenue":        c.get("trend_revenue"),
            "trend_margin":         c.get("trend_margin"),
            "risk_keywords":        risks,
            "rule_of_40_pass":      s.get("rule_of_40_pass"),
            "metrics_count":        len(m),
            "text_sections_filled": sum(1 for v in txt.values() if v and len(v) > 100),
        },
    }

# ─── MAIN PIPELINE ────────────────────────────────────────────────────────────

def process_company(company: dict, sess: EDGARSession, llm,
                    forms: tuple = ("10-K","10-Q","8-K"),
                    max_per_form: int = 8) -> list[dict]:
    ticker = company["ticker"]
    logger.info(f"══ {ticker} ({company['name']}) ══")

    cik = ticker_to_cik(ticker, sess)
    if not cik:
        return []

    xbrl = load_xbrl_facts(cik, sess)
    prev_metrics: dict[str, dict] = {}
    records: list[dict] = []

    for form_type in forms:
        filings = get_filings(cik, form_type, sess, max_per_form)

        for filing in filings:
            accno  = filing["accession_no"]
            period = filing["period_of_report"]
            prim   = filing["primary_document"]
            logger.info(f"  ▶ {form_type} | {period} | {accno}")

            rec_id = hashlib.md5(f"{ticker}_{form_type}_{period}_{accno}".encode()).hexdigest()[:12]
            base_meta = {
                "ticker": ticker, "company_name": company["name"], "cik": cik,
                "form_type": form_type, "filed_date": filing["filed_date"],
                "period_of_report": period,
                "fiscal_year": int(period[:4]) if period else None,
                "accession_no": accno,
            }

            # ── 8-K ──────────────────────────────────────────────────────
            if "8-K" in form_type:
                event_text = scrape_8k_text(cik, accno, prim, sess)
                record = {
                    "id": rec_id, "meta": base_meta,
                    "metrics": {}, "changes": {}, "signals": {},
                    "text_sections": {"event_text": event_text},
                }
                if event_text and len(event_text) > 300:
                    analysis = generate_analysis(record, llm)
                    if analysis:
                        record["instruction_pair"] = build_instruction_pair(record, analysis)
                        logger.info(f"    ✓ 8-K LLM: {len(analysis)} chars")
                records.append(record)
                continue

            # ── 10-K / 10-Q ──────────────────────────────────────────────
            metrics       = extract_metrics(xbrl, period, form_type) if xbrl else {}
            text_sections = scrape_text_sections(cik, accno, prim, form_type, sess)

            prev_key = f"{ticker}_{form_type}"
            changes  = compute_changes(metrics, prev_metrics[prev_key]) if prev_key in prev_metrics else {}
            if metrics:
                prev_metrics[prev_key] = metrics

            signals = compute_signals(metrics, changes)
            record  = {
                "id": rec_id, "meta": base_meta,
                "metrics": metrics, "changes": changes,
                "signals": signals, "text_sections": text_sections,
            }

            if len(metrics) >= 5:
                analysis = generate_analysis(record, llm)
                if analysis:
                    record["instruction_pair"] = build_instruction_pair(record, analysis)
                    has_text = any(v for v in text_sections.values() if v and len(v) > 100)
                    mode = "FULL" if has_text else "QUANT"
                    logger.info(f"    ✓ LLM [{mode}]: {len(analysis)} chars")
                else:
                    # BUG 5 FIX: mark for pending retry
                    record["_needs_llm"] = True
                    logger.warning(f"    ✗ LLM skipped — marked for retry")
            else:
                logger.warning(f"    ✗ LLM skipped: only {len(metrics)} metrics")

            records.append(record)

    logger.success(f"  {ticker}: {len(records)} records")
    return records


def retry_pending(pending_file: str, output_file: str, llm) -> int:
    """
    Retry LLM generation for records that hit TPD limit during main run.
    Reads pending file, generates analysis, appends to output file.
    """
    if not os.path.exists(pending_file):
        return 0

    pending = []
    with open(pending_file) as f:
        for line in f:
            try:
                r = json.loads(line)
                if r.get("_needs_llm"):
                    pending.append(r)
            except Exception:
                pass

    if not pending:
        return 0

    logger.info(f"Retrying LLM for {len(pending)} pending records...")
    completed = 0

    with open(output_file, "a", encoding="utf-8") as out:
        for record in tqdm(pending, desc="LLM retry"):
            record.pop("_needs_llm", None)
            analysis = generate_analysis(record, llm)
            if analysis:
                record["instruction_pair"] = build_instruction_pair(record, analysis)
                completed += 1
            out.write(json.dumps(record, ensure_ascii=False) + "\n")
            out.flush()

    os.remove(pending_file)
    logger.success(f"Retry complete: {completed}/{len(pending)} got LLM analysis")
    return completed


def build_dataset(tickers: list[dict], output_file: str = OUTPUT_FILE,
                  forms: tuple = ("10-K","10-Q","8-K"),
                  max_per_form: int = 8, resume: bool = True) -> int:
    sess = EDGARSession()
    llm  = get_llm()

    # Resume: skip already-processed tickers
    done: set[str] = set()
    if resume and os.path.exists(output_file):
        with open(output_file) as f:
            for line in f:
                try: done.add(json.loads(line)["meta"]["ticker"])
                except: pass
        if done:
            logger.info(f"Resuming — already done: {sorted(done)}")

    remaining = [c for c in tickers if c["ticker"] not in done]
    logger.info(f"Processing {len(remaining)}/{len(tickers)} companies | forms: {forms}")

    written  = 0
    pending  = 0

    with open(output_file, "a", encoding="utf-8") as out, \
         open(PENDING_FILE, "a", encoding="utf-8") as pf:
        for company in tqdm(remaining, desc="Companies"):
            try:
                records = process_company(
                    company=company, sess=sess, llm=llm,
                    forms=forms, max_per_form=max_per_form,
                )
                for rec in records:
                    if rec.get("_needs_llm"):
                        pf.write(json.dumps(rec, ensure_ascii=False) + "\n")
                        pf.flush()
                        pending += 1
                    else:
                        out.write(json.dumps(rec, ensure_ascii=False) + "\n")
                        out.flush()
                    written += 1
                logger.info(f"  ✓ {company['ticker']}: {len(records)} records | total: {written}")
            except Exception as e:
                logger.exception(f"  ✗ {company['ticker']} crashed: {e}")
                continue

    # Retry pending at end of run (TPD may have reset)
    if pending > 0:
        logger.info(f"\n{pending} records need LLM retry (TPD limit hit during run)")
        retried = retry_pending(PENDING_FILE, output_file, llm)
        logger.info(f"Retry result: {retried}/{pending} completed")

    logger.success(f"Done. {written} records → {output_file}")
    return written


In [10]:

# ─── ENTRY POINT ─────────────────────────────────────────────────────────────

if __name__ == "__main__":
    tickers = get_nasdaq100_tickers()
    tickers = tickers[:3]   # ← change to tickers or tickers[:N] for more

    print("=" * 62)
    print("SEC Fine-Tuning Dataset Builder  v8  (100% FREE)")
    print("=" * 62)
    print(f"  Companies      : {len(tickers)}")
    print(f"  Forms          : 10-K + 10-Q + 8-K")
    print(f"  Years back     : {YEARS_BACK}")
    print(f"  Output         : {OUTPUT_FILE}")
    print(f"  Log            : sec_builder.log")
    print(f"  LLM model      : {MODEL}")
    print(f"  TPD budget     : ~500k tokens/day (llama-4-maverick)")
    print(f"  GROQ key       : {'✓ set' if groq_api_key else '✗ MISSING'}")
    print(f"  Paid APIs      : NONE")
    print()
    print("FIXES vs v7:")
    print("  [BUG1] ✓ Largest .htm selected (not primaryDocument shell)")
    print("  [BUG2] ✓ 8-K: EX-99.1 press release fetched (not XBRL shell)")
    print("  [BUG3] ✓ TOC entries rejected (number-content quality gate)")
    print("  [BUG4] ✓ llama-4-maverick (500k TPD) + JSON output (~650 tokens)")
    print("  [BUG5] ✓ 429 retried with sleep, pending file for end-of-run retry")
    print("  [BUG6] ✓ XMLParsedAsHTMLWarning suppressed")
    print("=" * 62)

    if not groq_api_key:
        print("\nERROR: GROQ_API_KEY not set.")
        sys.exit(1)

    count = build_dataset(
        tickers=tickers,
        output_file=OUTPUT_FILE,
        forms=("10-K", "10-Q", "8-K"),
        max_per_form=8,
        resume=True,
    )
    print(f"\nDone: {count} records → {OUTPUT_FILE}")
    print(f"Log:  sec_builder.log")

12:42:04 | INFO    | Processing 3/3 companies | forms: ('10-K', '10-Q', '8-K')


SEC Fine-Tuning Dataset Builder  v8  (100% FREE)
  Companies      : 3
  Forms          : 10-K + 10-Q + 8-K
  Years back     : 5
  Output         : sec_finetune_dataset.jsonl
  Log            : sec_builder.log
  LLM model      : openai/gpt-oss-20b
  TPD budget     : ~500k tokens/day (llama-4-maverick)
  GROQ key       : ✓ set
  Paid APIs      : NONE

FIXES vs v7:
  [BUG1] ✓ Largest .htm selected (not primaryDocument shell)
  [BUG2] ✓ 8-K: EX-99.1 press release fetched (not XBRL shell)
  [BUG3] ✓ TOC entries rejected (number-content quality gate)
  [BUG4] ✓ llama-4-maverick (500k TPD) + JSON output (~650 tokens)
  [BUG5] ✓ 429 retried with sleep, pending file for end-of-run retry
  [BUG6] ✓ XMLParsedAsHTMLWarning suppressed


Companies:   0%|          | 0/3 [00:00<?, ?it/s]12:42:04 | INFO    | ══ AAPL (Apple Inc) ══
12:42:04 | INFO    | CIK table: 10,348 entries loaded
12:42:05 | INFO    |   XBRL: 503 GAAP + 2 DEI
12:42:05 | INFO    |   10-K: 5 filings
12:42:05 | INFO    |   ▶ 10-K | 2025-09-27 | 0000320193-25-000079
12:48:16 | INFO    |     Text sections: 4/6 filled
12:48:18 | INFO    |     ✓ LLM [FULL]: 1026 chars
12:48:18 | INFO    |   ▶ 10-K | 2024-09-28 | 0000320193-24-000123
12:54:04 | INFO    |     Text sections: 4/6 filled
12:54:06 | INFO    |     ✓ LLM [FULL]: 935 chars
12:54:06 | INFO    |   ▶ 10-K | 2023-09-30 | 0000320193-23-000106
12:57:41 | INFO    |     Text sections: 4/6 filled
12:57:43 | INFO    |     ✓ LLM [FULL]: 1110 chars
12:57:43 | INFO    |   ▶ 10-K | 2022-09-24 | 0000320193-22-000108
12:59:54 | INFO    |     Text sections: 4/6 filled
12:59:56 | INFO    |     ✓ LLM [FULL]: 1060 chars
12:59:56 | INFO    |   ▶ 10-K | 2021-09-25 | 0000320193-21-000105
13:01:27 | INFO    |     Text sectio


Done: 63 records → sec_finetune_dataset.jsonl
Log:  sec_builder.log
